# SAGA Quickstart Notebook

This notebook runs the full SAGA pipeline end-to-end on the synthetic mirror dataset
(Zenodo DOI `10.5281/zenodo.20260287`). It is the primary artifact for pipeline-level
replication of the SAGA manuscript results.

**Prerequisites:** Run `bash scripts/download_synthetic_mirror.sh` before executing
this notebook.

See [docs/reproducibility/quickstart.md](../docs/reproducibility/quickstart.md) for
expected wall-clock times and hardware requirements.

In [ ]:
import os
import numpy as np
import torch

import saga
from saga.config import SagaConfig
from saga.model.saga_model import SagaModel
from saga.utils.seed import set_all_seeds

print(f'SAGA version: {saga.__version__}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
SEED = 20260601
DATA_ROOT = '../data/synthetic'
CONFIG_PATH = '../configs/saga_main.yaml'

set_all_seeds(SEED)
config = SagaConfig.from_yaml(CONFIG_PATH)
print(f'Config loaded: d={config.model_dim}, L={config.num_layers}, H={config.num_heads}')

In [ ]:
model = SagaModel(config)
n_params = model.count_parameters()
print(f'SagaModel parameters: {n_params:,}')
assert abs(n_params - 10_872_960) < 50_000, (
    f'Parameter count {n_params} is far from the expected 10,872,960.'
)
print('Parameter count assertion PASSED.')

## Training

Training on the synthetic mirror (300,000 steps) takes approximately 15 hours on a single
A100 40 GB GPU. For a faster smoke test (50,000 steps), use:

```bash
bash scripts/run_quickstart_smoke.sh
```

The training loop is implemented in `src/saga/training/train_loop.py`. This notebook
cell is left as a placeholder; training is typically run from the command line.

In [ ]:
print('Training: run `bash scripts/train_saga.sh` from the repository root.')
print('See docs/reproducibility/quickstart.md for expected wall-clock times.')

## Evaluation

After training, run conformal calibration and inference:

```bash
bash scripts/calibrate_conformal.sh
bash scripts/run_inference.sh
```

Then verify the CRPS reduction against the headline manuscript value (31.9%):

```bash
bash scripts/verify_reproducibility.sh
```

In [ ]:
from saga.evaluation.metrics_probabilistic import crps_reduction_vs_baseline

crps_saga_manuscript = 0.318
crps_gkos_manuscript = 0.467
reduction = crps_reduction_vs_baseline(crps_saga_manuscript, crps_gkos_manuscript)
print(f'Manuscript CRPS reduction at h=10: {reduction:.1%}')